# Sympy code for FDM expansion of LBM diffusive equation up to Fourth-order

In [4]:
import sympy as sp
# -----------------------------------------------------------------------------
# 1) Symbols (parameters and step sizes)
# -----------------------------------------------------------------------------
dx, dt = sp.symbols('Delta_x Delta_t', positive=True, real=True)
s1, s2, w0, nu = sp.symbols('s_{1} s_{2} w_{0} \\nu', real=True)

def d(name, nx=0, nt=0):
    n = nx + nt
    if n == 1:
        num = rf"\partial{{{name}}}"
    else:
        num = rf"\partial^{{{n}}}{{{name}}}"
    den = ""
    if nx:
        den += rf"\partial{{x^{{{nx}}}}}" if nx > 1 else r"\partial{x}"
    if nt:
        den += rf"\partial{{t^{{{nt}}}}}" if nt > 1 else r"\partial{t}"
    return sp.Symbol(rf"\frac{{{num}}}{{{den}}}")
# -------------------1st-Order-Terms----------------------
dphit = d(r"\phi", nt=1); dphix   = d(r"\phi", nx=1)
# -------------------2nd-Order-Terms----------------------
dphix2 = d(r"\phi", nx=2); dphixt = d(r"\phi", nx=1, nt=1); dphit2 = d(r"\phi", nt=2)
# -------------------3rd-Order-Terms----------------------
dphix3 = d(r"\phi", nx=3); dphix2t = d(r"\phi", nx=2, nt=1); dphixt2 = d(r"\phi", nx=1, nt=2); dphit3  = d(r"\phi", nt=3)
# -------------------4th-Order-Terms----------------------
dphix4 = d(r"\phi", nx=4); dphix3t = d(r"\phi", nx=3, nt=1); dphix2t2 = d(r"\phi", nx=2, nt=2); dphixt3 = d(r"\phi", nx=1, nt=3); dphit4 = d(r"\phi", nt=4)

alpha1 = sp.simplify(1 - s1/2 - w0*s2/2)
alpha2 = sp.simplify((w0-1)*s2 + 1)
beta1  = sp.simplify(w0*s1*s2/2 - s1*s2/2 - w0*s2/2 + s1/2 +s2 -1)
beta2  = sp.simplify(-w0*s1*s2 + w0*s2 + s1 - 1)
gamma  = sp.simplify((s1-1)*(s2-1))
# -----------------------------------------------------------------------------
# 2) Continuous variables and field
# -----------------------------------------------------------------------------
x, t = sp.symbols('x t', real=True)
phi = sp.Function('phi')
# Shorthand for the base (continuous) field value at (x,t)
PHI = phi(x, t)

In [5]:
def taylor_phi(sx=0, st=0, order_x=4, order_t=4):
    expr = sp.S(0)
    # Keep all mixed terms up to the specified individual orders.
    for m in range(order_x + 1):
        for n in range(order_t + 1):
            # skip the (0,0) term? no, keep it.
            term = ( (sx*dx)**m * (st*dt)**n
                    / (sp.factorial(m)*sp.factorial(n))
                    * sp.diff(PHI, x, m, t, n) )
            expr += term
    return sp.expand(expr)

In [6]:
from IPython.display import display, Math
# phi_x^{t+1}, phi_x^{t}, phi_x^{t-1}, phi_x^{t-2}
phi_t_p1 = taylor_phi(sx=0,  st=+1, order_x=0, order_t=4)
display( Math(r"\phi_x^{t+1} = " + sp.latex(phi_t_p1)))
phi_t_0  = taylor_phi(sx=0,  st=0,  order_x=0, order_t=0)  # exactly phi(x,t)
display( Math(r"\phi_x^{t} = " + sp.latex(phi_t_0)))
phi_t_m1 = taylor_phi(sx=0,  st=-1, order_x=0, order_t=4)
display( Math(r"\phi_x^{t-1} = " + sp.latex(phi_t_m1)))
phi_t_m2 = taylor_phi(sx=0,  st=-2, order_x=0, order_t=4)
display( Math(r"\phi_x^{t-2} = " + sp.latex(phi_t_m2)))

# phi_{x±1}^{t}
phi_xm1_t0 = taylor_phi(sx=-1, st=0,  order_x=4, order_t=0)
display( Math(r"\phi_{x-1}^{t} = " + sp.latex(phi_xm1_t0)))
phi_xp1_t0 = taylor_phi(sx=+1, st=0,  order_x=4, order_t=0)
display( Math(r"\phi_{x+1}^{t} = " + sp.latex(phi_xp1_t0)))

# phi_{x±1}^{t-1}
phi_xm1_tm1 = taylor_phi(sx=-1, st=-1, order_x=4, order_t=4)
display( Math(r"\phi_{x-1}^{t-1} = " + sp.latex(phi_xm1_tm1)))
phi_xp1_tm1 = taylor_phi(sx=+1, st=-1, order_x=4, order_t=4)
display( Math(r"\phi_{x+1}^{t-1} = " + sp.latex(phi_xp1_tm1)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [7]:
# -----------------------------------------------------------------------------
# 5) Build the discrete equation (expanded) and move everything to one side
# -----------------------------------------------------------------------------
lhs = phi_t_p1

rhs = (alpha1*phi_xm1_t0 + alpha2*phi_t_0 + alpha1*phi_xp1_t0
       + beta1*phi_xm1_tm1 + beta2*phi_t_m1 + beta1*phi_xp1_tm1
       + gamma * phi_t_m2)

residual = sp.simplify(sp.expand(lhs - rhs))  # = 0 is the modified equation

In [11]:
max_dx=4
max_dt=2
max_total=4
expr = sp.expand(residual)
poly = sp.Poly(expr, dx, dt, domain='EX')  # keep symbolic coeffs
kept = sp.S(0)

for (px, pt), coeff in poly.terms():
    if (px <= max_dx) and (pt <= max_dt) and ((px + pt) <= max_total):
        kept += coeff * dx**px * dt**pt
kept

Delta_t**2*Delta_x**2*(-s_{1}*s_{2}*w_{0}*Derivative(phi(x, t), (t, 2), (x, 2))/4 + s_{1}*s_{2}*Derivative(phi(x, t), (t, 2), (x, 2))/4 - s_{1}*Derivative(phi(x, t), (t, 2), (x, 2))/4 + s_{2}*w_{0}*Derivative(phi(x, t), (t, 2), (x, 2))/4 - s_{2}*Derivative(phi(x, t), (t, 2), (x, 2))/2 + Derivative(phi(x, t), (t, 2), (x, 2))/2) + Delta_t**2*(-3*s_{1}*s_{2}*Derivative(phi(x, t), (t, 2))/2 + s_{1}*Derivative(phi(x, t), (t, 2)) + s_{2}*Derivative(phi(x, t), (t, 2))) + Delta_t*Delta_x**2*(s_{1}*s_{2}*w_{0}*Derivative(phi(x, t), t, (x, 2))/2 - s_{1}*s_{2}*Derivative(phi(x, t), t, (x, 2))/2 + s_{1}*Derivative(phi(x, t), t, (x, 2))/2 - s_{2}*w_{0}*Derivative(phi(x, t), t, (x, 2))/2 + s_{2}*Derivative(phi(x, t), t, (x, 2)) - Derivative(phi(x, t), t, (x, 2))) + Delta_t*s_{1}*s_{2}*Derivative(phi(x, t), t) + Delta_x**4*(-s_{1}*s_{2}*w_{0}*Derivative(phi(x, t), (x, 4))/24 + s_{1}*s_{2}*Derivative(phi(x, t), (x, 4))/24 + s_{2}*w_{0}*Derivative(phi(x, t), (x, 4))/12 - s_{2}*Derivative(phi(x, t), (x,

In [12]:
term1 = sp.expand(kept).subs({
    # ------------------------------------------------4th order--------------------------------------------------------------------------------------------
    PHI.diff(x, 4):dphix4,PHI.diff(x, 3, t):dphix3t,PHI.diff(x, 2, t, 2):dphix2t2,PHI.diff(x, t, 3):dphixt3,PHI.diff(t, 4):dphit4,
    # ------------------------------------------------3rd order--------------------------------------------------------------------------------------------
    PHI.diff(x, 3):dphix3, PHI.diff(x, 2, t):dphix2t, PHI.diff(x, t, 2):dphixt2, PHI.diff(t, 3):dphit3,
    # ------------------------------------------------2nd order--------------------------------------------------------------------------------------------
    PHI.diff(x, 2):dphix2, PHI.diff(x, t):dphixt, PHI.diff(t, 2):dphit2,
    # ------------------------------------------------1st order--------------------------------------------------------------------------------------------
    PHI.diff(x):dphix, PHI.diff(t):dphit,
})
display(term1)

-Delta_t**2*Delta_x**2*\frac{\partial^{4}{\phi}}{\partial{x^{2}}\partial{t^{2}}}*s_{1}*s_{2}*w_{0}/4 + Delta_t**2*Delta_x**2*\frac{\partial^{4}{\phi}}{\partial{x^{2}}\partial{t^{2}}}*s_{1}*s_{2}/4 - Delta_t**2*Delta_x**2*\frac{\partial^{4}{\phi}}{\partial{x^{2}}\partial{t^{2}}}*s_{1}/4 + Delta_t**2*Delta_x**2*\frac{\partial^{4}{\phi}}{\partial{x^{2}}\partial{t^{2}}}*s_{2}*w_{0}/4 - Delta_t**2*Delta_x**2*\frac{\partial^{4}{\phi}}{\partial{x^{2}}\partial{t^{2}}}*s_{2}/2 + Delta_t**2*Delta_x**2*\frac{\partial^{4}{\phi}}{\partial{x^{2}}\partial{t^{2}}}/2 - 3*Delta_t**2*\frac{\partial^{2}{\phi}}{\partial{t^{2}}}*s_{1}*s_{2}/2 + Delta_t**2*\frac{\partial^{2}{\phi}}{\partial{t^{2}}}*s_{1} + Delta_t**2*\frac{\partial^{2}{\phi}}{\partial{t^{2}}}*s_{2} + Delta_t*Delta_x**2*\frac{\partial^{3}{\phi}}{\partial{x^{2}}\partial{t}}*s_{1}*s_{2}*w_{0}/2 - Delta_t*Delta_x**2*\frac{\partial^{3}{\phi}}{\partial{x^{2}}\partial{t}}*s_{1}*s_{2}/2 + Delta_t*Delta_x**2*\frac{\partial^{3}{\phi}}{\partial{x^{2}}\

In [14]:
term2 = sp.collect(sp.expand(-term1/(dt*s1*s2))+ dphit,[dphit,dphix,
                                                         dphix2*dx*dx/dt,dphixt,dphit2*dt,
                                                         dphix2t*dx*dx,dphixt2,dphix3,dphit3*dt**2,
                                                         dphix2t2*dt*dx**2,dphix4*dx**4/dt,
                                                        ]).subs(dphix2t2,0)
display( Math(r"\frac{\partial\phi}{\partial t}= " + sp.latex(term2)))

<IPython.core.display.Math object>

In [29]:
nut=dx*dx*(w0/2-sp.Rational(1,2)-w0/s1+1/s1)/dt
term3 = sp.collect(sp.expand(term2.subs(nut,nu)
                       .subs(dphit2,nut*nut*dphix4)
                       .subs(dphix2t,nut*dphix4)), [dphix2*dx**2/dt,dphix4*dx**4/dt])
dphix4_term3=sp.simplify(sp.factor(term3.coeff(dphix4)).subs(dt*s1,(w0-1)*(2-s1)*dx**2/nu))
# display(dphix4_term3)
display( Math(r"\frac{\partial\phi}{\partial t}= " + sp.latex(term3.subs(term3.coeff(dphix4),dphix4_term3))))

<IPython.core.display.Math object>

In [15]:
# from sympy import latex
# print(latex(sp.simplify(kept.subs(dt**2*dx**2,0))))